# MMJL Capture and Notebook-State Regression Test

## 1. Markdown cell — test purpose and execution order

This notebook tests:

1. repository-root discovery from anywhere inside the repository
2. `src/` import setup
3. MMJL utility imports
4. Jupyter magic registration
5. literal Markdown and HTML logging
6. `%%jupy_capture`
7. the `%%jupy_tee` alias
8. failed notebook-state execution in visible order:
   `A -> B -> C`
9. successful recovery by moving backward:
   `B -> A`
10. stdout, exception, rich-display, and Matplotlib capture
11. `%jupy_file`
12. manifest inspection and validation
13. Markdown and HTML timeline generation
14. the Python analogs of `tree`, `wc -l`, and `cat`

The notebook-state dependency graph is:

```text
C creates numbers
B consumes numbers and creates squared_numbers
A consumes numbers and squared_numbers, computes a mean, and plots
```

The cells are displayed as:

```text
A
B
C
B again
A again
```

Therefore the first A and B should fail, C should succeed, and the
subsequent B and A should succeed.

---

# Looking further:

Here is the complete notebook sequence. The intended execution order is the notebook's ordinary top-to-bottom order, with the exception of the A → B → C → B → A example

---

**Let's go!!!**<br/><br/>

In [ ]:
## 2. Code cell 01 — locate the repository root and configure `sys.path`
import importlib
import os
import sys
import pathlib

starting_dir = pathlib.Path.cwd().resolve()
repo_root = None
src_path = None
package_path = None

for candidate_path in [starting_dir, *starting_dir.parents]:
    package_path = (
        candidate_path
        / "src"
        / "multimodal_jupy_logger"
    )

    if package_path.is_dir():
        repo_root = candidate_path
        break
    ##endof:  if package_path.is_dir()
##endof:  for candidate_path in [...]

if repo_root is None:
    raise RuntimeError(
        "Could not find a repository root containing "
        "src/multimodal_jupy_logger."
    )
##endof:  if repo_root is None

src_path = repo_root / "src"

os.chdir(repo_root)

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
##endof:  if str(src_path) not in sys.path

importlib.invalidate_caches()

print("starting_dir:", starting_dir)
print("repo_root:", repo_root)
print("src_path:", src_path)
print("current working directory:", pathlib.Path.cwd())
print("Python executable:", sys.executable)

In [ ]:
## 3. Code cell 02 — import MMJL and the path-display utilities
from multimodal_jupy_logger import (
    MultimodalJupyLogger,
    jupy_logger_register,
    register_jupy_logger,
)

from multimodal_jupy_logger.utils import (
    count_lines,
    path_display,
    print_file,
    tree,
)

from multimodal_jupy_logger.utils import path_display as dpypd

print("MMJL imports succeeded.")
print("path_display module:", dpypd)

In [ ]:
## 4. Code cell 03 — prove both utility import styles work

print("Directly imported function:")
print("  tree:", tree)

print("\nModule-namespace functions:")
print("  dpypd.tree:", dpypd.tree)
print("  dpypd.count_lines:", dpypd.count_lines)
print("  dpypd.print_file:", dpypd.print_file)

print("\nDirect and module attributes refer to the same functions:")
print("  tree is dpypd.tree:", tree is dpypd.tree)
print(
    "  count_lines is dpypd.count_lines:",
    count_lines is dpypd.count_lines,
)
print(
    "  print_file is dpypd.print_file:",
    print_file is dpypd.print_file,
)

In [ ]:
## 5. Code cell 04 — show the relevant repository tree

dpypd.tree(
    this_dir=repo_root,
    dirs_to_exclude=[
        ".git",
        "__pycache__",
        ".venv",
        ".venv_test_mmjl",
        ".ipynb_checkpoints",
    ],
    files_to_exclude=[
        ".pyc",
    ],
)

For:  **6**

Run this after restarting the kernel and loading the updated files.

```python
register_jupy_logger()
```

Expected registration text includes:

```text
%%jupy_log
%%jupy_capture
%%jupy_tee
```

In [ ]:
## 6. Code cell 05 — register the magics

register_jupy_logger()

In [ ]:
## 7. Code cell 06 — verify that the magics exist

ipython_shell = get_ipython()

magic_names = [
    "jupy_save",
    "jupy_file",
    "jupy_log",
    "jupy_capture",
    "jupy_tee",
    "jupy_markdown",
    "jupy_html",
    "jupy_inspect",
    "jupy_validate",
]

for magic_name in magic_names:
    line_magic = ipython_shell.find_line_magic(magic_name)
    cell_magic = ipython_shell.find_cell_magic(magic_name)

    print(
        f"{magic_name:16s}",
        f"line={line_magic is not None}",
        f"cell={cell_magic is not None}",
    )
##endof:  for magic_name in magic_names

## 8. Markdown cell — notebook-native hidden-text control

This cell tests Jupyter’s own Markdown rendering independently of MMJL.

```markdown
## Notebook-native `<details>` control

<details>
<summary>Click the arrow to reveal the notebook-native text</summary>

This text lives directly in a Jupyter Markdown cell.

If clicking the arrow reveals this paragraph, the notebook frontend is
rendering the HTML element correctly.

</details>
```

---

_DWB Note: I think we want to test the following:_

## Notebook-native `<details>` control

<details>
<summary>Click the arrow to reveal the notebook-native text</summary>

This text lives directly in a Jupyter Markdown cell.

If clicking the arrow reveals this paragraph, the notebook frontend is
rendering the HTML element correctly.

</details>

<br/><hr/>

For **9** (`Code cell 07 — log the original Markdown regression case`)

This preserves the original test: raw `<details>` HTML stored with the
`text/markdown` MIME type.

In [ ]:
%%jupy_log --label details-summary-markdown-regression --mime text/markdown
<details>
<summary>Click the arrow to reveal logged Markdown text</summary>

This content was logged as `text/markdown`.

The regression question is whether exported timelines preserve this as
renderable Markdown/HTML or incorrectly turn it into escaped or fenced
source text.

</details>

## 9 Code cell 07 — log the original Markdown regression case

For **10** (`Code cell 08 — log an HTML control version`)

This distinguishes a general `<details>` failure from a Markdown-MIME
rendering-policy failure.

With the current timeline builder, the expected distinction is:

* `text/html`: should render as a collapsible block in the HTML timeline
* `text/markdown`: currently may appear as escaped/fenced source

That would identify a timeline-rendering issue, not a capture failure.

In [ ]:
%%jupy_log --label details-summary-html-control --mime text/html
<details>
<summary>Click the arrow to reveal logged HTML text</summary>

<p>
This content was logged as <code>text/html</code>. It should be inserted
as HTML in the generated HTML timeline.
</p>

</details>

## 10. Code cell 08 — log an HTML control version

In [ ]:
# Code cell 09

dpypd.tree(
    this_dir=repo_root,
    dirs_to_exclude=[
        ".git",
        "__pycache__",
        ".venv",
        ".venv_test_mmjl",
        ".ipynb_checkpoints",
    ],
    files_to_exclude=[
        ".pyc",
    ],
)

In [ ]:
## Code cell 10

dpypd.tree("C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log")

In [ ]:
##  Code cell 11 — ANYTHING NEW? (reusable count_lines or print_file)
##+      MAKE SURE TO MATCH THE FILENAME WITH TREE OUTPUT
print("##############################################################")
print(); print(
    "----------------------------------------------------------\n" + \
    "___File line count for___\n" + \
    "   'jupy_log/\n" + \
    "    \n" + \
    "    '___\n" + \
    "----------------------------------------------------------"
); dpypd.count_lines(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        ""
        ""
    )
); print("----------------------------------------------------------")
print("=========================================================="); print()
print("==========================================================")
print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000002_1781709912766_2026-06-17T112512766-0400_\n" + \
    "    details-summary-html-control.html'___\n" + \
    "----------------------------------------------------------"
); dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/"
        ""
    )
); print("----------------------------------------------------------")
print(); print()
print("##############################################################")

#### New stuff after Code cell 11



---


### Making this easier for next time

---

```python
Code cell N-1

dpypd.tree("C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log")
```

---

```python
##  Code cell N — Reusable combined tree and (count_lines or print_file)

##+    CHANGE THINGS IF THE FILENAMES DON'T MATCH THE TIMESTAMPS
##+    AND TAGS FROM dpypd.tree
##+ Should probably be made into a function, but lean-to execution,
##+ quick and reckless, get it done, artifacts. --DWB
print("##############################################################")
#print(); print(
#    "----------------------------------------------------------"\n + \
#    "___Output of___\n" + "dpypd.tree(\n" + \
#    "    \"C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log\"\n" + \
#    ")___" + \
#    "----------------------------------------------------------"
#); dpypd.tree("C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log")
#print("----------------------------------------------------------"); print()
#print("=========================================================="); print()
#print(); print("==========================================================")
print(); print(
print(); print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000001_1781709909178_2026-06-17T112509178-0400_\n" + \
    "    details-summary-markdown-regression.md'___\n" + \
    "----------------------------------------------------------"
); dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/000001_1781709909178_2026-06-17T112509178-0400_"
        "details-summary-markdown-regression.md"
    )
); print("----------------------------------------------------------")
print("=========================================================="); print()
print("==========================================================")
print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000002_1781709912766_2026-06-17T112512766-0400_\n" + \
    "    details-summary-html-control.html'___\n" + \
    "----------------------------------------------------------"
); dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/000002_1781709912766_2026-06-17T112512766-0400_"
        "details-summary-html-control.html"
    )
); print("----------------------------------------------------------")
print(); print()
print("##############################################################")
```

In [ ]:
## Code cell 12, MAKE SURE TO MATCH THE FILENAME WITH TREE OUTPUT

print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000001_1781709909178_2026-06-17T112509178-0400_\n" + \
    "    details-summary-markdown-regression.md'___\n" + \
    "----------------------------------------------------------"
)

dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/000001_1781709909178_2026-06-17T112509178-0400_"
        "details-summary-markdown-regression.md"
    )
)

print("----------------------------------------------------------")

In [ ]:
## Code cell 13, MAKE SURE TO MATCH THE FILENAME WITH TREE OUTPUT

print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000002_1781709912766_2026-06-17T112512766-0400_\n" + \
    "    details-summary-html-control.html'___\n" + \
    "----------------------------------------------------------"
)

dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/000002_1781709912766_2026-06-17T112512766-0400_"
        "details-summary-html-control.html"
    )
)

print("----------------------------------------------------------")

In [ ]:
##  Code cell 14 — ANYTHING NEW? (reusable count_lines or print_file)
##+      MAKE SURE TO MATCH THE FILENAME WITH TREE OUTPUT
print("##############################################################")
print(); print(
    "----------------------------------------------------------\n" + \
    "___File line count for___\n" + \
    "   'jupy_log/\n" + \
    "    \n" + \
    "    '___\n" + \
    "----------------------------------------------------------"
); dpypd.count_lines(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        ""
        ""
    )
); print("----------------------------------------------------------")
print("=========================================================="); print()
print("==========================================================")
print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000002_1781709912766_2026-06-17T112512766-0400_\n" + \
    "    details-summary-html-control.html'___\n" + \
    "----------------------------------------------------------"
); dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/"
        ""
    )
); print("----------------------------------------------------------")
print(); print()
print("##############################################################")

#### New stuff after Code cell 14



---


For **11** (`Code cell 15 — basic ``%%jupy_capture`` smoke test`)

Expected capture roles include:

```text
input
stdout
display
```

The final expression `5` may be represented through one or more MIME
items, commonly including `text/plain`.

In [ ]:
%%jupy_capture --label capture-basic
message = "This stdout should be displayed and logged."
print(message)

## 11. Code cell 15 — basic `%%jupy_capture` smoke test

In [ ]:
## Code cell 16

dpypd.tree("C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log")

In [ ]:
##  Code cell 17 — ANYTHING NEW? (reusable count_lines or print_file)
##+      MAKE SURE TO MATCH THE FILENAME WITH TREE OUTPUT
print("##############################################################")
print(); print(
    "----------------------------------------------------------\n" + \
    "___File line count for___\n" + \
    "   'jupy_log/\n" + \
    "    \n" + \
    "    '___\n" + \
    "----------------------------------------------------------"
); dpypd.count_lines(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        ""
        ""
    )
); print("----------------------------------------------------------")
print("=========================================================="); print()
print("==========================================================")
print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000002_1781709912766_2026-06-17T112512766-0400_\n" + \
    "    details-summary-html-control.html'___\n" + \
    "----------------------------------------------------------"
); dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/"
        ""
    )
); print("----------------------------------------------------------")
print(); print()
print("##############################################################")

#### New stuff after Code cell 17



---


For **12** (`Code cell 18 — basic ``%%jupy_tee`` alias test`)

No comments.

In [ ]:
%%jupy_tee --label tee-basic
tee_message = "The %%jupy_tee alias executed this cell."
print(tee_message)

tee_message.upper()

## 12. Code cell 18 — basic `%%jupy_tee` alias test

In [ ]:
## Code cell 19

dpypd.tree("C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log")

In [ ]:
##  Code cell 20 — ANYTHING NEW? (reusable count_lines or print_file)
##+      MAKE SURE TO MATCH THE FILENAME WITH TREE OUTPUT
print("##############################################################")
print(); print(
    "----------------------------------------------------------\n" + \
    "___File line count for___\n" + \
    "   'jupy_log/\n" + \
    "    \n" + \
    "    '___\n" + \
    "----------------------------------------------------------"
); dpypd.count_lines(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        ""
        ""
    )
); print("----------------------------------------------------------")
print("=========================================================="); print()
print("==========================================================")
print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000002_1781709912766_2026-06-17T112512766-0400_\n" + \
    "    details-summary-html-control.html'___\n" + \
    "----------------------------------------------------------"
); dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/"
        ""
    )
); print("----------------------------------------------------------")
print(); print()
print("##############################################################")

#### New stuff after Code cell 20



---


In [ ]:
%matplotlib inline

## 13.1. Code cell 21

### -^- Doing this early, because I want to. For **13.1**.

## 13. Begin the notebook-state test

### 14. Description and preparation: A, B, C, then B, A

The next five capture cells must be run in their displayed order first, then partially reversed:

```text
A first
B first
C
B second
A second
```

Expected results:

```text
A first  -> NameError because squared_numbers does not exist
B first  -> NameError because numbers does not exist
C        -> creates numbers successfully
B second -> creates squared_numbers successfully
A second -> computes the mean and creates the plot successfully
```

The reset cell immediately below, **14** (`Code cell 22 — reset hidden kernel state`), removes any old values that might make the deliberately incorrect first pass appear to work.

In [ ]:
%%jupy_capture --label abc-state-reset
state_names = [
    "numbers",
    "squared_numbers",
    "avg_value",
    "fig",
    "ax",
    "plot_source_path",
]

removed_names = []

for state_name in state_names:
    if state_name in globals():
        globals().pop(state_name)
        removed_names.append(state_name)
    ##endof:  if state_name in globals()
##endof:  for state_name in state_names

print("Removed prior state:", removed_names)
print("A and B should now fail on their first executions.")

## 14. Code cell 22 — reset hidden kernel state

In [ ]:
## Code cell 23

dpypd.tree("C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log")

In [ ]:
##  Code cell 24 — ANYTHING NEW? (reusable count_lines or print_file)
##+      MAKE SURE TO MATCH THE FILENAME WITH TREE OUTPUT
print("##############################################################")
print(); print(
    "----------------------------------------------------------\n" + \
    "___File line count for___\n" + \
    "   'jupy_log/\n" + \
    "    \n" + \
    "    '___\n" + \
    "----------------------------------------------------------"
); dpypd.count_lines(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        ""
        ""
    )
); print("----------------------------------------------------------")
print("=========================================================="); print()
print("==========================================================")
print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000002_1781709912766_2026-06-17T112512766-0400_\n" + \
    "    details-summary-html-control.html'___\n" + \
    "----------------------------------------------------------"
); dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/"
        ""
    )
); print("----------------------------------------------------------")
print(); print()
print("##############################################################")

#### New stuff after Code cell 24



---


**`Cell A, below`**
- For **15** (`Code cell         17 — A, first execution: expected failure`)

  - Expected result:

```text
NameError involving squared_numbers
```

  - The capture should still log:

    * the input
    * the exception
    * any output produced before the exception, if present

  - ACTUAL OUTPUT:

```text

```

  - Output of `dpypd.tree()

  - Relevant content of `FILENAME` manifest

```

```

  - Content of `Filename1` markdown

```

```

  - Content of `Filename1` text

```

```

- For **20** (`Code cell (still) 17 — A, second execution: expected success and plot capture`)

  - The only difference in the input is the change,
    - from:
      - `%%jupy_capture --label abc-first-pass-a`
    - to:
      - `%%jupy_capture --label abc-second-pass-a`
  
  - ACTUAL OUTPUT
 
```

```

The rest of the successful run's files' contents can be seen in the output of Code cell
    

In [ ]:
%%jupy_capture --label abc-first-pass-a
##done before#import matplotlib.plotly as plt
## P.S. kamMA's Jupyter must not need a numpy import(?)

avg_value = sum(squared_numbers) / len(squared_numbers)

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(
    numbers,
    squared_numbers,
    marker="o",
)

ax.axhline(
    avg_value,
    linestyle="--",
    label=f"mean squared value = {avg_value:.2f}",
)

ax.set_title(
    "Forwards-wrong / backwards-right notebook-state test"
)
ax.set_xlabel("number")
ax.set_ylabel("number squared")
ax.grid(True)
ax.legend()

plt.show()

## 15. Code cell         25 — A, first execution: expected failure
#
## and
#
## 20. Code cell (still) 25 — A, second execution: expected success and plot capture

In [ ]:
## Code cell 26

dpypd.tree("C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log")

In [ ]:
##  Code cell 27 — Reusable combined tree and (count_lines or print_file)

##+    CHANGE THINGS IF THE FILENAMES DON'T MATCH THE TIMESTAMPS
##+    AND TAGS FROM dpypd.tree
##+ i.e. MAKE SURE TO MATCH THE FILENAME WITH TREE OUTPUT
##+ Should probably be made into a function, but lean-to execution,
##+ quick and reckless, get it done, artifacts. --DWB
print("##############################################################")
print(); print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000001_1781709909178_2026-06-17T112509178-0400_\n" + \
    "    details-summary-markdown-regression.md'___\n" + \
    "----------------------------------------------------------"
); dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/000001_1781709909178_2026-06-17T112509178-0400_"
        "details-summary-markdown-regression.md"
    )
); print("----------------------------------------------------------")
print("=========================================================="); print()
print("==========================================================")
print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000002_1781709912766_2026-06-17T112512766-0400_\n" + \
    "    details-summary-html-control.html'___\n" + \
    "----------------------------------------------------------"
); dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/000002_1781709912766_2026-06-17T112512766-0400_"
        "details-summary-html-control.html"
    )
); print("----------------------------------------------------------")
print(); print()
print("##############################################################")

In [ ]:
##  Code cell 28 — ANYTHING NEW? (reusable count_lines or print_file)
##+      MAKE SURE TO MATCH THE FILENAME WITH TREE OUTPUT
print("##############################################################")
print(); print(
    "----------------------------------------------------------\n" + \
    "___File line count for___\n" + \
    "   'jupy_log/\n" + \
    "    \n" + \
    "    '___\n" + \
    "----------------------------------------------------------"
); dpypd.count_lines(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        ""
        ""
    )
); print("----------------------------------------------------------")
print("=========================================================="); print()
print("==========================================================")
print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000002_1781709912766_2026-06-17T112512766-0400_\n" + \
    "    details-summary-html-control.html'___\n" + \
    "----------------------------------------------------------"
); dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/"
        ""
    )
); print("----------------------------------------------------------")
print(); print()
print("##############################################################")

#### New stuff after Code cell 28



---


**`Cell B, below`**
- For **16** (`Code cell         29 — first execution: expected failure`)

  - Expected result:

```text
NameError involving numbers
```

  - File contents for unsuccessful run
    - Look at the output of Code cell 
  - ACTUAL OUTPUT

- For **18** (`Code cell (still) 29 — second execution: expected success`)

  - The only difference in the input is the change,
    - from:
      - `%%jupy_capture --label abc-first-pass-b`
    - to:
      - `%%jupy_capture --label abc-second-pass-b`

In [ ]:
%%jupy_capture --label abc-first-pass-b
squared_numbers = [
    number ** 2
    for number in numbers
]

print("squared_numbers:", squared_numbers)

## 16. Code cell         29 — B, first execution: expected failure
#
## and
#
## 18. Code cell (still) 29 — B, second execution: expected success

In [ ]:
## Code cell 30

dpypd.tree("C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log")

In [ ]:
##  Code cell 31 — Reusable combined tree and (count_lines or print_file)
##+    CHANGE THINGS IF THE FILENAMES DON'T MATCH THE TIMESTAMPS
##+    AND TAGS FROM dpypd.tree
##+ i.e. MAKE SURE TO MATCH THE FILENAME WITH TREE OUTPUT
##+ Should probably be made into a function, but lean-to execution,
##+ quick and reckless, get it done, artifacts. --DWB
print("##############################################################")
print(); print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000001_1781709909178_2026-06-17T112509178-0400_\n" + \
    "    details-summary-markdown-regression.md'___\n" + \
    "----------------------------------------------------------"
); dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/000001_1781709909178_2026-06-17T112509178-0400_"
        "details-summary-markdown-regression.md"
    )
); print("----------------------------------------------------------")
print("=========================================================="); print()
print("==========================================================")
print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000002_1781709912766_2026-06-17T112512766-0400_\n" + \
    "    details-summary-html-control.html'___\n" + \
    "----------------------------------------------------------"
); dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/000002_1781709912766_2026-06-17T112512766-0400_"
        "details-summary-html-control.html"
    )
); print("----------------------------------------------------------")
print(); print()

print("##############################################################")

In [ ]:
##  Code cell 32 — ANYTHING NEW? (reusable count_lines or print_file)
##+      MAKE SURE TO MATCH THE FILENAME WITH TREE OUTPUT
print("##############################################################")
print(); print(
    "----------------------------------------------------------\n" + \
    "___File line count for___\n" + \
    "   'jupy_log/\n" + \
    "    \n" + \
    "    '___\n" + \
    "----------------------------------------------------------"
); dpypd.count_lines(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        ""
        ""
    )
); print("----------------------------------------------------------")
print("=========================================================="); print()
print("==========================================================")
print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000002_1781709912766_2026-06-17T112512766-0400_\n" + \
    "    details-summary-html-control.html'___\n" + \
    "----------------------------------------------------------"
); dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/"
        ""
    )
); print("----------------------------------------------------------")
print(); print()
print("##############################################################")

#### New stuff after Code cell 32



---


**`Cell C, below`**
- For **17** (`Code cell 33 — C: expected success`)

  - Expected result:
    - Success. 
  - File contents for successful run
    - Look at the output of Code cell 
  - ACTUAL OUTPUT

```

```


In [ ]:
%%jupy_capture --label abc-cell-c
import numpy as np
import matplotlib.pyplot as plt

# numbers = [
#     -5,
#     -3,
#     -1,
#     0,
#     2,
#     4,
#     6,
# ]

## Older global-random-state style, like MLU
# numbers = np.random.randint(1, 11, size=10)


rng = np.random.default_rng()
numbers = rng.integers(1, 11, size=10)

print("numbers:", numbers)

## 17. Code cell 34 — C: expected success

In [ ]:
## Code cell 35

dpypd.tree("C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log")

In [ ]:
##  Code cell 36 — Reusable combined tree and (count_lines or print_file)
##+    CHANGE THINGS IF THE FILENAMES DON'T MATCH THE TIMESTAMPS
##+    AND TAGS FROM dpypd.tree
##+ i.e. MAKE SURE TO MATCH THE FILENAME WITH TREE OUTPUT
##+ Should probably be made into a function, but lean-to execution,
##+ quick and reckless, get it done, artifacts. --DWB
print("##############################################################")
print(); print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000001_1781709909178_2026-06-17T112509178-0400_\n" + \
    "    details-summary-markdown-regression.md'___\n" + \
    "----------------------------------------------------------"
); dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/000001_1781709909178_2026-06-17T112509178-0400_"
        "details-summary-markdown-regression.md"
    )
); print("----------------------------------------------------------")
print("=========================================================="); print()
print("==========================================================")
print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000002_1781709912766_2026-06-17T112512766-0400_\n" + \
    "    details-summary-html-control.html'___\n" + \
    "----------------------------------------------------------"
); dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/000002_1781709912766_2026-06-17T112512766-0400_"
        "details-summary-html-control.html"
    )
); print("----------------------------------------------------------")
print(); print()
print("##############################################################")

In [ ]:
##  Code cell 37 — ANYTHING NEW? (reusable count_lines or print_file)
##+      MAKE SURE TO MATCH THE FILENAME WITH TREE OUTPUT
print("##############################################################")
print(); print(
    "----------------------------------------------------------\n" + \
    "___File line count for___\n" + \
    "   'jupy_log/\n" + \
    "    \n" + \
    "    '___\n" + \
    "----------------------------------------------------------"
); dpypd.count_lines(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        ""
        ""
    )
); print("----------------------------------------------------------")
print("=========================================================="); print()
print("==========================================================")
print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000002_1781709912766_2026-06-17T112512766-0400_\n" + \
    "    details-summary-html-control.html'___\n" + \
    "----------------------------------------------------------"
); dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/"
        ""
    )
); print("----------------------------------------------------------")
print(); print()
print("##############################################################")

#### New stuff after Code cell 37



---


(**18** combined with **16** — both are Cell B)

## 19.~~Code cell~~ — ~~configure Matplotlib inline rendering~~ 

#### Note: trying (later: tried and ???ed) to do this earlier, in **13.1**

~~Run this before the successful A cell.~~

~~\`\`\``python`~~<br/>
~~`%matplotlib inline`~~<br/>
~~\`\`\`~~

---

(**20** combined with **16** — both are Cell A)

---

## For **15**&ndash;**20**, looking towards **21**

This tests two related but distinct things:

* inline PNG capture by `%%jupy_capture`
* creation of a file that `%jupy_file` can log explicitly

In [ ]:
## Code cell 38

dpypd.tree("C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log")

In [ ]:
##  Code cell 39 — Reusable combined tree and (count_lines or print_file)
##+    CHANGE THINGS IF THE FILENAMES DON'T MATCH THE TIMESTAMPS
##+    AND TAGS FROM dpypd.tree
##+ i.e. MAKE SURE TO MATCH THE FILENAME WITH TREE OUTPUT
##+ Should probably be made into a function, but lean-to execution,
##+ quick and reckless, get it done, artifacts. --DWB
print("##############################################################")
print(); print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000001_1781709909178_2026-06-17T112509178-0400_\n" + \
    "    details-summary-markdown-regression.md'___\n" + \
    "----------------------------------------------------------"
); dpypd.count_lines(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/000001_1781709909178_2026-06-17T112509178-0400_"
        "details-summary-markdown-regression.md"
    )
); print("----------------------------------------------------------")
print("=========================================================="); print()
print("==========================================================")
print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000002_1781709912766_2026-06-17T112512766-0400_\n" + \
    "    details-summary-html-control.html'___\n" + \
    "----------------------------------------------------------"
); dpypd.count_lines(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/000002_1781709912766_2026-06-17T112512766-0400_"
        "details-summary-html-control.html"
    )
); print("----------------------------------------------------------")
print(); print()
print("##############################################################")

In [ ]:
##  Code cell 40 — ANYTHING NEW? (reusable count_lines or print_file)
##+      MAKE SURE TO MATCH THE FILENAME WITH TREE OUTPUT
print("##############################################################")
print(); print(
    "----------------------------------------------------------\n" + \
    "___File line count for___\n" + \
    "   'jupy_log/\n" + \
    "    \n" + \
    "    '___\n" + \
    "----------------------------------------------------------"
); dpypd.count_lines(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        ""
        ""
    )
); print("----------------------------------------------------------")
print("=========================================================="); print()
print("==========================================================")
print(
    "----------------------------------------------------------\n" + \
    "___File contents for___\n" + \
    "   'jupy_log/artifacts/\n" + \
    "    000002_1781709912766_2026-06-17T112512766-0400_\n" + \
    "    details-summary-html-control.html'___\n" + \
    "----------------------------------------------------------"
); dpypd.print_file(
    (
        "C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/"
        "artifacts/"
        ""
    )
); print("----------------------------------------------------------")
print(); print()
print("##############################################################")

#### New stuff after Code cell 40



---

Not doing so many details

For **21** (`Code cell 41 — test ``%jupy_file`` with the saved plot`)

The magic currently appends `-01` to the supplied label because it
supports logging multiple paths in one call.

In [ ]:
%jupy_file --label abc-explicit-plot --mime image/png jupy_log/staging/abc_backwards_right.png

## 21. Code cell 41 — test `%jupy_file` with the saved plot

In [ ]:
## 22. Code cell 42 — ~~initial~~ manifest inspection and validation

%jupy_inspect

print("\n" + "-" * 72 + "\n")

%jupy_validate

For **23** (`## Code cell 43 — instantiate a direct logger for programmatic inspection`)

This logger points at the same root as the registered magics.

In [ ]:
## 23. Code cell 43 — instantiate a direct logger for programmatic inspection

log_root = repo_root / "jupy_log"
logger = MultimodalJupyLogger(root=log_root)

manifest_rows = logger.read_manifest_rows()

print("log_root:", log_root)
print("manifest:", logger.manifest)
print("artifact directory:", logger.artifacts)
print("manifest rows:", len(manifest_rows))

For **24** (`## Code cell 44 — inspect the most recent manifest rows`)

Look for groups corresponding to:

```text
details-summary-markdown-regression
details-summary-html-control
capture-basic
tee-basic
abc-state-reset
abc-first-pass-a
abc-first-pass-b
abc-cell-c
abc-second-pass-b
abc-second-pass-a
abc-explicit-plot
```

In [ ]:
## 24. Code cell 44 — inspect the most recent manifest rows

recent_row_count = 20
recent_rows = manifest_rows[-recent_row_count:]

for row in recent_rows:
    print(
        row.get("artifact_sequence", ""),
        "capture=" + (row.get("capture_sequence", "") or "-"),
        "role=" + row.get("role", ""),
        "mime=" + row.get("mime", ""),
        "label=" + row.get("label", ""),
    )
##endof:  for row in recent_rows

In [ ]:
## 25. Code cell 45 — inspect the log tree with `dpypd.tree`

dpypd.tree(
    this_dir=log_root,
    dirs_to_exclude=[
        "__pycache__",
    ],
    files_to_exclude=[
        ".pyc",
    ],
)

In [ ]:
## 26. Code cell 46 — use the `wc -l` analog

manifest_line_count = dpypd.count_lines(
    logger.manifest,
    do_print=True,
)

print(
    "Expected manifest data rows:",
    manifest_line_count - 1,
)
print(
    "Rows returned by read_manifest_rows:",
    len(manifest_rows),
)
print(
    "Counts agree:",
    manifest_line_count - 1 == len(manifest_rows),
)

For **27** (`Code cell 47 — use the ``cat`` analog on the manifest`)

Verify visually that:

* `artifact_sequence` increases monotonically
* capture artifacts share a `capture_sequence`
* the first failed A and B have exception rows
* the later B and A have successful output rows
* filenames contain both the sequence and epoch milliseconds

In [ ]:
dpypd.print_file(
    logger.manifest,
    show_line_numbers=True,
)

For **28** (`Code cell 48 — use the ``cat`` analog on the manifest`)

The equivalent magic calls are:

```python
%jupy_markdown
%jupy_html
```

Using the direct logger here makes the returned paths easy to retain.

In [ ]:
## 28. Code cell 48 — generate both timelines

markdown_timeline_path = logger.build_markdown()
html_timeline_path = logger.build_html()

print("Markdown timeline:", markdown_timeline_path)
print("HTML timeline:", html_timeline_path)

In [ ]:
## 29. Code cell 49 — count timeline lines

dpypd.count_lines(
    markdown_timeline_path,
    do_print=True,
)

dpypd.count_lines(
    html_timeline_path,
    do_print=True,
)

For **30** (`Code cell 50 — print the Markdown timeline`)

Things to check:

* artifact and capture sequences appear
* roles appear
* failed A and B occur before successful C, B, and A
* Python input is preserved
* exception text is preserved
* image links are present
* the Markdown `<details>` regression item is probably fenced rather than rendered

In [ ]:
## 30. Code cell 50 — print the Markdown timeline

dpypd.print_file(
    markdown_timeline_path,
    show_line_numbers=False,
)

For **31** (`Code cell 51 — inspect the raw HTML timeline source`)

This may be lengthy, but it is useful for this first pass.

Search visually for:

```text
details-summary-markdown-regression
details-summary-html-control
abc-first-pass-a
abc-first-pass-b
abc-second-pass-a
image/png
```

For **32** (`Code cell 52 — render the generated HTML timeline in the notebook`)

Regression expectations:

* the `text/html` control should show a clickable `<details>` arrow
* the `text/markdown` version may show escaped `<details>` source
* the successful A plot should appear if its captured `image/png` path is
  usable from the generated timeline
* exception entries should be visible in chronological artifact order

In [ ]:
## 32. Code cell — render the generated HTML timeline in the notebook

from IPython.display import HTML, display

display(
    HTML(
        filename=str(html_timeline_path),
    )
)

In [ ]:
## 33. Code cell — final validation after exports

missing_paths = logger.validate_manifest()

print("\nFinal status:")
print("  manifest rows:", len(logger.read_manifest_rows()))
print("  missing artifact paths:", len(missing_paths))
print("  Markdown timeline exists:", markdown_timeline_path.exists())
print("  HTML timeline exists:", html_timeline_path.exists())
print("  explicit plot source exists:", plot_source_path.exists())

In [ ]:
## 34. Code cell — final ordering assertions

final_rows = logger.read_manifest_rows()

artifact_sequences = [
    int(row["artifact_sequence"])
    for row in final_rows
]

sequences_are_monotonic = (
    artifact_sequences
    == sorted(artifact_sequences)
)

sequences_are_unique = (
    len(artifact_sequences)
    == len(set(artifact_sequences))
)

filenames_have_sequence_prefixes = all(
    pathlib.Path(row["path"]).name.startswith(
        f"{int(row['artifact_sequence']):06d}_"
    )
    for row in final_rows
)

print("Artifact sequences monotonic:", sequences_are_monotonic)
print("Artifact sequences unique:", sequences_are_unique)
print(
    "Filenames begin with artifact sequence:",
    filenames_have_sequence_prefixes,
)

## 35. Markdown cell — record the result

### Test-result notes

Record the observations here:

- [ ] repository-root discovery worked
- [ ] imports worked
- [ ] utility module alias worked
- [ ] individual utility imports worked
- [ ] magics registered
- [ ] notebook-native `<details>` worked
- [ ] `text/markdown` regression behavior observed
- [ ] `text/html` control rendered
- [ ] `%%jupy_capture` executed and logged
- [ ] `%%jupy_tee` executed and logged
- [ ] first A failed as expected
- [ ] first B failed as expected
- [ ] C succeeded
- [ ] second B succeeded
- [ ] second A succeeded
- [ ] inline Matplotlib plot was captured
- [ ] explicit `%jupy_file` plot was logged
- [ ] manifest sequences were monotonic
- [ ] capture groups were visible
- [ ] filenames contained millisecond timestamps
- [ ] validation reported no missing artifacts
- [ ] Markdown timeline was generated
- [ ] HTML timeline was generated
- [ ] timeline execution history was reconstructable